# derived_8.4-eval-mlp-2.1 — optimize the mixed family + fair SWA re-test + finish the 96-family debias (~1 h H100 wall)

Follow-up to `derived_8.4-eval-mlp-2.0` (the `2regime_mixed` family broke the plain-MLP ceiling: honest val top-5 ensemble test R² 0.8003, 2-seed honest single 0.7903; `fg`/`plr` documented negatives; the 2.0 SWA recipe a negative with two prescribed fixes). 2.1 is an **optimization + further parameter sweep**, sized to spend ~1 h of the 2 h H100 wall allocation (340 job-seeds at 8 workers):

1. **Fair SWA re-test** — `swa_start_frac` swept {0.7, 0.75, 0.8, 0.85} + an **RNG guard** around the BN-recalibration pass so a swa job's *live* trajectory is bit-identical to its anchor (`mlp21/trainer.py`, the two 2.0-prescribed fixes).
2. **Denser seed coverage** — a 3-phase, 3-seed sweep {42, 7, 123} to beat the 2.0 val-noise finding (mixed Spearman(val, test) = -0.455); per-family phase depth (the mixed winner gets the densest).
3. **No fg/plr re-spend** — documented negatives get no GPU; all 178 phase-1 configs are plain MLP.
4. **96-family debias** — grid the small-net neighborhood (width 96–256, dropout 0.4–0.6, lr {3e-4, 1e-3}, huber, mixup, late SWA) to push the 96-pool families' median bias²/MSE below 5 %.

Protocol unchanged and honest: train on train (2017–2020, n=9,803), early-stop/select on official val (2021–2022, n=4,805), evaluate on untouched test (2023–2025, n=6,620); multi-seed mean val RMSE selection among the mlp/fg/plr winner pool (mlp-only in practice); aux2020 diagnostic only; patience-60 kept; no calibration / no trainval retrain (documented negatives). LOSO out of scope (per the 2.0 brief).

All numbers below are the stdout of this executed notebook. Weights/checkpoints/test predictions under `models/`; preprocessed tensors and per-job logs under `artifacts/`; figures at the experiment root.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json

# Robust resolution of the experiment dir whether executed from notebooks/ or in-place.
candidates = [Path.cwd() / "experiment/derived_8.4-eval-mlp-2.1", Path.cwd()]
EXP_DIR = next((p for p in candidates if (p / "metrics_summary.csv").exists()), Path.cwd())

df_summary = pd.read_csv(EXP_DIR / "metrics_summary.csv")
df_per_regime = pd.read_csv(EXP_DIR / "per_regime_metrics_summary.csv")
df_sweep = pd.read_csv(EXP_DIR / "sweep_results.csv")
df_timing = pd.read_csv(EXP_DIR / "timing_summary.csv")
df_bias = pd.read_csv(EXP_DIR / "bias_summary.csv") if (EXP_DIR / "bias_summary.csv").exists() else None
df_bias_cl = pd.read_csv(EXP_DIR / "bias_by_cluster.csv") if (EXP_DIR / "bias_by_cluster.csv").exists() else None
df_ood = pd.read_csv(EXP_DIR / "ood_summary.csv") if (EXP_DIR / "ood_summary.csv").exists() else None
df_stop20 = pd.read_csv(EXP_DIR / "stopping_20_summary.csv") if (EXP_DIR / "stopping_20_summary.csv").exists() else None
df_stop20_agg = pd.read_csv(EXP_DIR / "stopping_20_aggregate.csv") if (EXP_DIR / "stopping_20_aggregate.csv").exists() else None
df_sel = pd.read_csv(EXP_DIR / "selection_summary.csv") if (EXP_DIR / "selection_summary.csv").exists() else None
df_swa_meta = pd.read_csv(EXP_DIR / "swa_seed_meta.csv") if (EXP_DIR / "swa_seed_meta.csv").exists() else None
df_swa_ident = pd.read_csv(EXP_DIR / "swa_bit_identity.csv") if (EXP_DIR / "swa_bit_identity.csv").exists() else None
with open(EXP_DIR / "selected_features.json") as f:
    selected_meta = json.load(f)
with open(EXP_DIR / "timing_log.json") as f:
    timing_log = json.load(f)

fam_labels = {"2regime_96": "2-Regime-96", "2regime_54": "2-Regime-54", "2regime_mixed": "2-Regime-Mixed"}
print("loaded", len(df_summary), "leaderboard rows,", len(df_sweep), "sweep rows")

loaded 55 leaderboard rows, 178 sweep rows


## Selection Protocol v8 Diagnostic

Selection = multi-seed mean val RMSE among the honest architectures (mlp / fg / plr — 2.1 runs only mlp configs, so the pool is mlp-only in practice). 2.1 adds a 3rd seed (123) for the top configs per family (3-phase sweep) because 2.0 documented that the mixed family's val ranking is noisy (Spearman(val, test) = -0.455). aux2020 stays diagnostic-only (measures train fit). This section reports the val ranking, the Spearman correlations vs test at 1-/2-/3-seed aggregation, and the phase-stability table from `analyze_selection.py`.


In [2]:
from scipy.stats import spearmanr
HONEST = ("mlp", "fg", "plr")
print("### Selection Protocol v8 Diagnostic (selection = multi-seed mean val RMSE; mlp/fg/plr pool)")
for family, fam_label in fam_labels.items():
    sub = df_sweep[df_sweep["family"] == family].dropna(subset=["test_r2"]).copy()
    if sub.empty:
        continue
    sub = sub.sort_values("val_rmse", na_position="last").reset_index(drop=True)
    print(f"\n#### {fam_label} — top-10 by val RMSE")
    cols = ["config_id", "architecture", "n_seeds", "val_rmse", "aux_rmse", "test_r2", "test_rmse", "test_bias"]
    print(sub.head(10)[cols].to_markdown(index=False))
    valid = sub.dropna(subset=["val_rmse", "test_r2"])
    if len(valid) >= 8:
        rho, p = spearmanr(valid["val_rmse"], valid["test_r2"])
        print(f"  Spearman(val_rmse, test_r2) = {rho:+.3f} (p={p:.3f}, n={len(valid)})")
    honest_sub = sub[sub["architecture"].isin(HONEST)]
    val_honest = honest_sub.sort_values("val_rmse").iloc[0]
    test_best = sub.sort_values("test_r2", ascending=False).iloc[0]
    print(f"  val winner (honest) : {val_honest['config_id']} (test_r2={val_honest['test_r2']:.4f})")
    print(f"  test best (ref)     : {test_best['config_id']} (test_r2={test_best['test_r2']:.4f})")

if df_sel is not None:
    print("\n### Selection-reliability summary (analyze_selection.py)")
    print("#### Spearman(val, test) by aggregation depth")
    sub = df_sel[df_sel["aggregation"].str.startswith(("1-seed", "2-seed", "3-seed"))]
    print(sub.to_markdown(index=False))
    print("\n#### Phase stability — winner at each seed depth")
    print(df_sel[df_sel["aggregation"].str.startswith("winner|")].to_markdown(index=False))


### Selection Protocol v8 Diagnostic (selection = multi-seed mean val RMSE; mlp/fg/plr pool)

#### 2-Regime-96 — top-10 by val RMSE
| config_id                      | architecture   |   n_seeds |   val_rmse |   aux_rmse |   test_r2 |   test_rmse |   test_bias |
|:-------------------------------|:---------------|----------:|-----------:|-----------:|----------:|------------:|------------:|
| w512x512x512_d0.3_lr1e-3       | mlp            |         3 |  0.0484652 |  0.0260292 |  0.759483 |   0.0499584 |   0.0188537 |
| w512x512x512_d0.3_huber0.1_swa | mlp            |         3 |  0.0492833 |  0.0251614 |  0.753595 |   0.0505662 |   0.0216425 |
| w256x256_d0.4_lr1e-3           | mlp            |         3 |  0.0507216 |  0.0326816 |  0.71615  |   0.0542726 |   0.0297628 |
| w192x192_d0.5_lr1e-3           | mlp            |         3 |  0.0508919 |  0.0338218 |  0.725573 |   0.0533641 |   0.0287853 |
| w256x256_d0.5_lr1e-3_swa075    | mlp            |         3 |  0.0509407 |  0.0321609 

## Overall Model Leaderboard

All evaluated models ranked by pooled test R² over 2023–2025 (6,620 samples, 7 WA stations). MLP rows carry the sweep `config_id` and `n_seeds`; `(val top-k avg)` rows are offline seed-averaged ensembles of the top-k val-selected honest configs (no extra training); `(5-seed champ, ...)` rows are 5-seed champion ensembles of the val-selected winners (extra stability seeds, no trainval retrain — documented negative); `cross-family` rows average the val-selected winners across families. XGBoost rows are the eval-1.1 references; `MLP-1.3` / `MLP-2.0` rows are the previous experiments' val-selected winners + test-best references (2.0's mixed val top-5 ensemble 0.8003 is the number 2.1 must beat); `test-best` rows are reporting only (selection on test would be leakage).

In [3]:
cols = ["model_name", "strategy_name", "pooled_r2", "pooled_rmse", "pooled_ubrmse", "pooled_bias", "pooled_mae", "pooled_pearson"]
print("### Overall Leaderboard (2023-2025 Test Set)")
print(df_summary[cols].to_markdown(index=False))

### Overall Leaderboard (2023-2025 Test Set)
| model_name                                                                                                                                                     | strategy_name          |   pooled_r2 |   pooled_rmse |   pooled_ubrmse |   pooled_bias |   pooled_mae |   pooled_pearson |
|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:-----------------------|------------:|--------------:|----------------:|--------------:|-------------:|-----------------:|
| Clustering_V0_Full_k2 (Winner c0=0, c1=10)                                                                                                                     | XGBoost_Reference      |    0.81496  |     0.0438196 |       0.043337  |    0.00648567 |    0.0337195 |         0.905594 |
| MLP-2.0 2-Regime-Mixed (val top-5 avg)                                                         

## Hyperparameter Sweep Summary

178 curated phase-1 configs (all `mlp` — `fg`/`plr` are documented negatives from 2.0 and get no GPU) generated deterministically by `make_configs.py` from 2-factor grids around the 2.0 winners: mixed shape × lr / loss × dropout / wd × mixup / act × depth / lr × huber / SWA start-frac × configs; 96 small-net width × dropout × lr / huber / mixup / SWA; 54 width × lr / loss × act / width × huber / SWA. 8 parallel H100 workers; configs ranked by **multi-seed mean val RMSE** (honest signal); test R² for reference. Phase-2 configs carry `n_seeds=2`, phase-3 configs `n_seeds=3`.

In [4]:
for family, fam_label in fam_labels.items():
    sub = df_sweep[df_sweep["family"] == family].sort_values("val_rmse", na_position="last").head(10)
    print(f"### Sweep Top-10 — {fam_label} (by val RMSE, the honest selection signal)")
    cols = ["config_id", "architecture", "n_seeds", "dropout", "lr", "loss", "val_rmse", "aux_rmse", "test_r2", "test_rmse", "test_bias", "best_epoch", "train_time_s"]
    show = sub[cols].copy()
    show["deployed"] = [json.loads((EXP_DIR / "models" / family / cid / "meta.json").read_text()).get("deployed", "live")
                        if (EXP_DIR / "models" / family / cid / "meta.json").exists() else "" for cid in sub["config_id"]]
    print(show.to_markdown(index=False))
    print()

### Sweep Top-10 — 2-Regime-96 (by val RMSE, the honest selection signal)
| config_id                      | architecture   |   n_seeds |   dropout |     lr | loss   |   val_rmse |   aux_rmse |   test_r2 |   test_rmse |   test_bias |   best_epoch |   train_time_s | deployed   |
|:-------------------------------|:---------------|----------:|----------:|-------:|:-------|-----------:|-----------:|----------:|------------:|------------:|-------------:|---------------:|:-----------|
| w512x512x512_d0.3_lr1e-3       | mlp            |         3 |       0.3 | 0.001  | mse    |  0.0484652 |  0.0260292 |  0.759483 |   0.0499584 |   0.0188537 |          263 |        122.383 | live       |
| w512x512x512_d0.3_huber0.1_swa | mlp            |         3 |       0.3 | 0.0003 | huber  |  0.0492833 |  0.0251614 |  0.753595 |   0.0505662 |   0.0216425 |          260 |        153.728 | live       |
| w256x256_d0.4_lr1e-3           | mlp            |         3 |       0.4 | 0.001  | mse    |  0.0507216 |

## Per-Regime Performance Breakdown

Cluster 0 holds 73 % of the test rows, so it dominates the pooled R². Per-cluster test metrics for the val top-3 honest configs per family (the mixed family's c1 = 54+10 specialist is expected to hold the ~0.83 R² of the 54-family's c1 while c0 gains the 96-pool fit), the XGBoost references, and the 1.3 / 2.0 reference winners.

In [5]:
print("### Per-Regime Performance Breakdown")
cols = ["strategy_name", "model_name", "cluster", "n_train", "n_test", "r2", "rmse", "ubrmse", "bias", "mae"]
print(df_per_regime[cols].to_markdown(index=False))

### Per-Regime Performance Breakdown
| strategy_name     | model_name                                              |   cluster |   n_train |   n_test |       r2 |      rmse |    ubrmse |         bias |       mae |
|:------------------|:--------------------------------------------------------|----------:|----------:|---------:|---------:|----------:|----------:|-------------:|----------:|
| MLP_2regime_96    | MLP 2-Regime-96 (w512x512x512_d0.3_lr1e-3)              |         0 |      7156 |     4817 | 0.751308 | 0.0498896 | 0.0471617 |  0.0162712   | 0.0390899 |
| MLP_2regime_96    | MLP 2-Regime-96 (w512x512x512_d0.3_lr1e-3)              |         1 |      2647 |     1803 | 0.77822  | 0.0501416 | 0.0430228 |  0.0257531   | 0.0377413 |
| MLP_2regime_96    | MLP 2-Regime-96 (w512x512x512_d0.3_huber0.1_swa)        |         0 |      7156 |     4817 | 0.750949 | 0.0499257 | 0.0460893 |  0.0191925   | 0.0392907 |
| MLP_2regime_96    | MLP 2-Regime-96 (w512x512x512_d0.3_huber0.1_swa)        

## Yearly Performance Breakdown

Year-by-year R² on the 2023–2025 test period. 2.0 fixed the historically weak 2025 year for the mixed family's ensembles (2025 R² 0.8336 — best of any model); this table tracks whether the 2.1 winners hold that year.

In [6]:
year_cols = [c for c in df_summary.columns if c.startswith("year_") and c.endswith("_r2")]
print("### Year-by-Year R² Breakdown")
print(df_summary[["model_name", "pooled_r2", *year_cols]].to_markdown(index=False))

### Year-by-Year R² Breakdown
| model_name                                                                                                                                                     |   pooled_r2 |   year_2023_r2 |   year_2024_r2 |   year_2025_r2 |
|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|------------:|---------------:|---------------:|---------------:|
| Clustering_V0_Full_k2 (Winner c0=0, c1=10)                                                                                                                     |    0.81496  |       0.822971 |       0.783256 |       0.83029  |
| MLP-2.0 2-Regime-Mixed (val top-5 avg)                                                                                                                         |    0.800323 |       0.745352 |       0.825612 |       0.832424 |
| MLP 2-Regime-Mixed (test-best, w448x448x448_d0.3_huber0.

## Systematic-Bias Diagnostic (headline)

mlp-1.2/1.3 documented the 96-family's systematic positive test bias (bias² ≈ 10–17 % of MSE); 2.0 got the 54-family under 5 % (median 3.7 %) but the 96-pool families still carry 10–13 %. 2.1's debias lever is capacity control at scale: the small-net grid (width 96–256, dropout 0.4–0.6, lr up to 1e-3) targets the 2.0 finding that <200k-param nets are near-unbiased (`w256x256_d0.5`: bias²/MSE 1.0 %, test R² 0.7854). Success criterion: per-family median bias²/MSE < 5 % for ALL three families. Backed by `analyze_bias.py`.

In [7]:
if df_bias is not None:
    print("### Per-family median bias^2/MSE share (honest architectures)")
    print("| family     | n_configs |   med_bias2_mse_share |   med_test_bias |   med_test_r2 |")
    print("|:-----------|----------:|----------------------:|----------------:|--------------:|")
    for fam, fam_label in fam_labels.items():
        sub = df_bias[(df_bias["family"] == fam) & df_bias["architecture"].isin(HONEST)]
        if sub.empty:
            continue
        print(f"| {fam} | {len(sub)} | {sub['bias2_mse_share'].median():.4f} | "
              f"{sub['test_bias'].median():.4f} | {sub['test_r2'].median():.4f} |")
    cols = ["family", "config_id", "architecture", "test_r2", "test_rmse", "test_bias", "bias2_mse_share"]
    print("\n### Worst 8 configs by bias^2/MSE share (all architectures)")
    print(df_bias.sort_values("bias2_mse_share", ascending=False).head(8)[cols].to_markdown(index=False))
    print("\n### Best 8 configs by bias^2/MSE share (all architectures)")
    print(df_bias.sort_values("bias2_mse_share").head(8)[cols].to_markdown(index=False))
    if df_bias_cl is not None:
        print("\n### Per-cluster median bias^2/MSE share (honest architectures)")
        print("| family     | cluster |   med_bias2_mse_share |   med_test_bias |   med_test_r2 |")
        print("|:-----------|--------:|----------------------:|----------------:|--------------:|")
        for fam, fam_label in fam_labels.items():
            for cl in (0, 1):
                sub = df_bias_cl[(df_bias_cl["family"] == fam) & (df_bias_cl["cluster"] == cl)
                                 & df_bias_cl["architecture"].isin(HONEST)]
                if sub.empty:
                    continue
                print(f"| {fam} | {cl} | {sub['bias2_mse_share'].median():.4f} | "
                      f"{sub['test_bias'].median():.4f} | {sub['test_r2'].median():.4f} |")
else:
    print("bias_summary.csv not found — run analyze_bias.py after the sweep.")

### Per-family median bias^2/MSE share (honest architectures)
| family     | n_configs |   med_bias2_mse_share |   med_test_bias |   med_test_r2 |
|:-----------|----------:|----------------------:|----------------:|--------------:|
| 2regime_96 | 46 | 0.1386 | 0.0190 | 0.7394 |
| 2regime_54 | 32 | 0.0080 | 0.0033 | 0.7800 |
| 2regime_mixed | 100 | 0.1274 | 0.0172 | 0.7614 |

### Worst 8 configs by bias^2/MSE share (all architectures)
| family        | config_id                     | architecture   |   test_r2 |   test_rmse |   test_bias |   bias2_mse_share |
|:--------------|:------------------------------|:---------------|----------:|------------:|------------:|------------------:|
| 2regime_mixed | w768x768_d0.3_huber0.1_lr1e-3 | mlp            |  0.646923 |   0.06053   |   0.0377072 |          0.388068 |
| 2regime_mixed | w640x640_d0.3_huber0.1_lr1e-3 | mlp            |  0.640656 |   0.0610649 |   0.0379793 |          0.386822 |
| 2regime_mixed | w256x256_d0.3_huber0.1_lr6e-4 | mlp 

## SWA vs Live Deployment — the fair re-test (trainer)

2.0's SWA section documented a negative: with `swa_start_frac: 0.6` equal-weight averaging over epochs 240–400 + BN recalibration, no SWA snapshot ever beat the live best on val, and the recalibration leaked RNG into the live trajectory (gains un-attributable). 2.1 applies the two prescribed fixes in `mlp21/trainer.py`:

- **(a) `swa_start_frac` swept {0.7, 0.75, 0.8, 0.85}** (0.85 × 400 = 340 risks never starting before the patience-60 stop; the sweep tests all four values).
- **(b) RNG guard** (`_rng_guard` around the BN-recalibration forward pass) — a swa job's *live* val curve must now be **bit-identical** to its non-SWA anchor's (the stack check below proves it).

Deployment stays honest: the SWA snapshot is deployed iff its best val RMSE beats the live best (within-val comparison, per seed and per specialist). Backed by `analyze_swa.py` (`swa_seed_meta.csv` + `swa_bit_identity.csv`).

In [8]:
if df_swa_meta is not None:
    swa = df_swa_meta[df_swa_meta["swa"]]
    def _swa_mean(s):
        # specialists that early-stopped before swa_start_epoch keep inf in
        # their per-seed meta; mean over the started ones only, else NaN.
        import numpy as _np
        vals = [float(v) for v in s if _np.isfinite(v)]
        return float(_np.mean(vals)) if vals else float("nan")
    agg = (swa.groupby(["family", "config_id"])
              .agg(n_seeds=("seed", "nunique"),
                   n_specs=("cluster", "size"),
                   n_deployed_swa=("deployed", lambda s: int((s == "swa").sum())),
                   val_rmse_live=("val_rmse_live", "mean"),
                   val_rmse_swa=("val_rmse_swa", _swa_mean),
                   swa_start_frac=("swa_start_frac", "first"))
              .reset_index())
    def _fmt(v):
        import numpy as _np
        return "n/a" if not _np.isfinite(v) else f"{v:.5f}"
    print("### Per-config SWA deployment (per-seed per-cluster live vs SWA val)")
    print("| family | config_id | swa_start_frac | n_seeds | n_specs | specs_deployed_swa | val_rmse_live | val_rmse_swa |")
    print("|:---|---:|---:|---:|---:|---:|---:|---:|")
    for _, r in agg.sort_values(["family", "config_id"]).iterrows():
        print(f"| {r['family']} | {r['config_id']} | {r['swa_start_frac']} | {r['n_seeds']} | {r['n_specs']} | "
              f"{r['n_deployed_swa']} | {_fmt(r['val_rmse_live'])} | {_fmt(r['val_rmse_swa'])} |")
    n_deployed = int((swa["deployed"] == "swa").sum())
    print()
    print(f"**Deployment verdict:** {n_deployed}/{len(swa)} (seed, specialist) jobs deployed the SWA snapshot.")
else:
    print("swa_seed_meta.csv not found — run analyze_swa.py after the sweep.")

if df_swa_ident is not None and not df_swa_ident.empty:
    n_ident = int(df_swa_ident["bit_identical"].sum())
    print()
    print(f"### Bit-identity stack check (RNG-guard proof): {n_ident}/{len(df_swa_ident)} "
          f"(swa-live vs anchor) val-curve pairs bit-identical (max|diff| < 1e-12).")
    bad = df_swa_ident[~df_swa_ident["bit_identical"]]
    if not bad.empty:
        print("NON-IDENTICAL pairs (guard broken):")
        print(bad.to_string(index=False))
    else:
        print("All compared pairs bit-identical — swa gains are attributable to SWA, not RNG drift.")
else:
    print("swa_bit_identity.csv not found — run analyze_swa.py after the sweep.")


### Per-config SWA deployment (per-seed per-cluster live vs SWA val)
| family | config_id | swa_start_frac | n_seeds | n_specs | specs_deployed_swa | val_rmse_live | val_rmse_swa |
|:---|---:|---:|---:|---:|---:|---:|---:|
| 2regime_54 | w384x384_d0.3_gelu_swa075 | 0.75 | 1 | 2 | 0 | 0.05640 | 0.14212 |
| 2regime_54 | w384x384_d0.3_gelu_swa085 | 0.85 | 1 | 2 | 0 | 0.05640 | n/a |
| 2regime_54 | w448x448_d0.3_gelu_swa075 | 0.75 | 2 | 4 | 0 | 0.05587 | n/a |
| 2regime_54 | w448x448_d0.3_gelu_swa085 | 0.85 | 3 | 6 | 0 | 0.07438 | 0.13445 |
| 2regime_54 | w512x512x512_d0.3_huber0.1_swa075 | 0.75 | 3 | 6 | 0 | 0.05153 | 0.09777 |
| 2regime_54 | w512x512x512_d0.3_huber0.1_swa085 | 0.85 | 3 | 6 | 0 | 0.05153 | 0.08672 |
| 2regime_96 | w128x128_d0.5_swa075 | 0.75 | 2 | 4 | 0 | 0.04899 | 0.08479 |
| 2regime_96 | w128x128_d0.5_swa085 | 0.85 | 2 | 4 | 0 | 0.04899 | 0.08448 |
| 2regime_96 | w192x192_d0.5_swa075 | 0.75 | 2 | 4 | 0 | 0.05010 | 0.09773 |
| 2regime_96 | w192x192_d0.5_swa085 | 0.85 | 2

## FeatureGroupedMLP / PLR — documented negatives, not re-run in 2.1

2.0 established that the grouped-tower (`fg`, best 0.782) and PLR-encoding (`plr`, best 0.720) architectures underperform the plain MLP (0.790) at this scale — the winning lever was the *feature allocation* (the `2regime_mixed` family), not the tower structure. Per the no-re-spend rule, **2.1 runs no fg/plr configs**; the classes and the validated semantic grouping remain available in `mlp21/feature_groups.py`. The grouping table for the union of the three families' features is printed for reference.

In [9]:
import sys as _sys
_sys.path.insert(0, str(EXP_DIR))
from mlp21.feature_groups import summary_table
data = selected_meta
union_feats = sorted(set(selected_meta.get("shared_backbone_54", [])) | set(selected_meta.get("candidate_pool_96", [])) | set(selected_meta.get("cluster_1_delta_features", [])))
print(f"Union of the 3 families' features: {len(union_feats)}")
print(summary_table(list(union_feats)))

Union of the 3 families' features: 116
| group_id | group | n_features | features |
|---|---|---|---|
| 0 | smap | 26 | A_d_SMAP_sm_interp_kobs14, A_d_SMAP_sm_interp_kobs30, A_grad_SMAP_sm_interp_kobs14, A_grad_SMAP_sm_interp_kobs30, A_grad_SMAP_sm_interp_kobs7, C_lag_SMAP_sm_interp_kobs12, C_lag_SMAP_sm_interp_kobs30, SMAP_ampm_diff_interp, SMAP_sm_am_interp, SMAP_sm_am_interp_lag1, SMAP_sm_am_interp_lag30, SMAP_sm_am_interp_rollrange30, SMAP_sm_am_interp_rollrange7, SMAP_sm_interp_lag7, SMAP_sm_interp_rollrange30, SMAP_sm_interp_rollrange7, SMAP_sm_pm_interp, SMAP_sm_pm_interp_lag1, SMAP_sm_pm_interp_lag30, SMAP_sm_pm_interp_lag7, SMAP_sm_pm_interp_rollmean30, SMAP_sm_pm_interp_rollrange30, SMAP_sm_pm_interp_rollrange7, V_ema_SMAP_sm_interp_kobs30, V_rollmin_SMAP_sm_interp_kobs14, V_rollmin_SMAP_sm_interp_kobs30 |
| 1 | optical | 7 | A_grad_s2_b11_kobs30, V_rollmin_s2_b11_kobs14, V_rollmin_s2_b11_kobs30, V_rollmin_s2_b12_kobs30, V_rollrng_s2_b11_kobs30, s2_b4, s2_b8 |
| 2 | vegetatio

## Early-Stopping & SWA-rule Replay (patience-60 re-check)

Offline replay of honest epoch-selection rules on the saved per-epoch curves (`analyze_stopping.py --tag 20`). 1.2/1.3 established that patience-60 is the best honest rule; 2.0 additionally replayed the `swa_val` rule on the SWA snapshot curves. 2.1 re-checks all of it on the new (RNG-guarded, later-start) SWA curves — if late-start SWA produces a usable SWA-val curve, the replay shows whether selecting the SWA epoch beats patience-60.

In [10]:
if df_stop20_agg is not None:
    print("### Stopping-rule aggregates (mean pooled test RMSE; lower is better; oracle = unreachable bound)")
    print(df_stop20_agg.to_markdown(index=False))
else:
    print("stopping_20_aggregate.csv not found — run analyze_stopping.py --tag 20 after the sweep.")

### Stopping-rule aggregates (mean pooled test RMSE; lower is better; oracle = unreachable bound)
| family        | rule             |   mean_test_rmse |   median_test_rmse |   n |
|:--------------|:-----------------|-----------------:|-------------------:|----:|
| 2regime_96    | patience60       |        0.0526684 |          0.0529503 |  90 |
| 2regime_96    | patience20       |        0.0526684 |          0.0529503 |  90 |
| 2regime_96    | patience40       |        0.0526684 |          0.0529503 |  90 |
| 2regime_96    | val_aux          |        0.0545995 |          0.0545689 |  90 |
| 2regime_96    | swa_val          |        0.0561794 |          0.0538431 |  90 |
| 2regime_96    | plateau_w20e1e-4 |        0.0760381 |          0.0772552 |  90 |
| 2regime_96    | plateau_w40e1e-4 |        0.0686752 |          0.0657465 |  90 |
| 2regime_96    | plateau_w40e3e-4 |        0.0686752 |          0.0657465 |  90 |
| 2regime_96    | plateau_w60e1e-4 |        0.0601043 |          0.05540

## Extrapolation (OOD) Check

588/6,620 test rows (8.9 %) are OOD on ≥1 top-10 gain feature (same definition as mlp-1.0–2.0). The pure-96 family keeps its OOD strength; 2.0 showed the mixed family is in-distribution-strong but OOD-weak (its c1 = 54+10 half carries the 54-family's weak OOD). The 2.1 winners' OOD behavior is reported for the record (family allocation is pinned, so this is a tracking table, not a target).

In [11]:
if df_ood is not None:
    print("### Extrapolation check (OOD test slices)")
    print(df_ood.to_markdown(index=False))
else:
    print("ood_summary.csv not found — run analyze_extrapolation.py after the sweep.")

### Extrapolation check (OOD test slices)
| model                                                  | slice           |    n |       r2 |      rmse |        bias |       mae |
|:-------------------------------------------------------|:----------------|-----:|---------:|----------:|------------:|----------:|
| MLP 2regime-96 (w512x512x512_d0.3_lr1e-3)              | all             | 6620 | 0.759483 | 0.0499584 |  0.0188537  | 0.0387226 |
| MLP 2regime-96 (w512x512x512_d0.3_lr1e-3)              | in_distribution | 6032 | 0.753545 | 0.0514519 |  0.0213449  | 0.0401715 |
| MLP 2regime-96 (w512x512x512_d0.3_lr1e-3)              | ood             |  588 | 0.757848 | 0.0306936 | -0.0067026  | 0.0238591 |
| MLP 2regime-96 (5-seed champ)                          | all             | 6620 | 0.75656  | 0.050261  |  0.0204007  | 0.0391776 |
| MLP 2regime-96 (5-seed champ)                          | in_distribution | 6032 | 0.749983 | 0.0518224 |  0.0227929  | 0.0407259 |
| MLP 2regime-96 (5-seed ch

## Overfitting-Symptom Analysis

From the saved artifacts (no retraining), via `analyze_overfitting.py`: train-fit vs held-out gap (aux2020 = train-fit), capacity vs test transfer, and the per-epoch curve shape for each family's val winner. 2.0's winners show the familiar pattern (test min early, val flat, train-fit improving); the 2.1 multi-seed selection and (for the 96-family) the small-net capacity control are the mitigations in play.

In [12]:
print("### Overfitting symptoms (analyze_overfitting.py)")
# The CLI writes overfitting_summary.csv; recompute the key tables inline.
import sys as _sys
if str(EXP_DIR) not in _sys.path:
    _sys.path.insert(0, str(EXP_DIR))
from analyze_overfitting import compute_overfitting
r = compute_overfitting(df_sweep, EXP_DIR)
print("\n#### 1. Train-fit vs held-out gap (median RMSE)")
print("| family     |   aux2020 (train-fit) |   val |   test |   val/train ratio |")
print("|:-----------|----------------------:|------:|-------:|------------------:|")
for fam in fam_labels:
    print(f"| {fam} | {r[f'{fam}_med_aux']:.4f} | {r[f'{fam}_med_val']:.4f} | "
          f"{r[f'{fam}_med_test']:.4f} | {r[f'{fam}_train_val_ratio']:.1f}x |")
print("\n#### 2. Capacity vs test transfer (median by n_params bucket)")
print("| family     | capacity   |   n_configs |   med_val_rmse |   med_test_r2 |   med_test_bias |")
print("|:-----------|:-----------|------------:|---------------:|--------------:|----------------:|")
for fam in fam_labels:
    for row in r[f"{fam}_capacity"]:
        print(f"| {row['family']} | {row['capacity']} | {row['n_configs']} | "
              f"{row['med_val_rmse']:.4f} | {row['med_test_r2']:.4f} | {row['med_test_bias']:.4f} |")
print("\n#### 3. Per-epoch curve shape for the val winner (cluster-0 specialist)")
print("| family     | config_id |   aux_ep100 |   aux_ep260 |   val_plateau |   test_min |   test_min_epoch |   test_at_best_val |   test_final |   test_rise_after_min |")
print("|:-----------|:----------|------------:|------------:|--------------:|-----------:|-----------------:|-------------------:|-------------:|----------------------:|")
for row in r["curve_rows"]:
    print(f"| {row['family']} | {row['config_id']} | {row['aux_ep100']:.4f} | {row['aux_ep260']:.4f} | "
          f"{row['val_plateau']:.4f} | {row['test_min']:.4f} | {row['test_min_epoch']} | "
          f"{row['test_at_best_val']:.4f} | {row['test_final']:.4f} | {row['test_rise_after_min']:.4f} |")
print("\n#### 4. Systematic bias on test (MLP vs XGBoost references)")
for fam in fam_labels:
    print(f"MLP {fam} median test bias: {r[f'{fam}_med_test_bias']:.4f}")
print("XGBoost references (eval-1.1): 2-regime 0.0065, global 0.0105")


### Overfitting symptoms (analyze_overfitting.py)

#### 1. Train-fit vs held-out gap (median RMSE)
| family     |   aux2020 (train-fit) |   val |   test |   val/train ratio |
|:-----------|----------------------:|------:|-------:|------------------:|
| 2regime_96 | 0.0422 | 0.0532 | 0.0520 | 1.3x |
| 2regime_54 | 0.0305 | 0.0613 | 0.0478 | 2.0x |
| 2regime_mixed | 0.0284 | 0.0511 | 0.0498 | 1.8x |

#### 2. Capacity vs test transfer (median by n_params bucket)
| family     | capacity   |   n_configs |   med_val_rmse |   med_test_r2 |   med_test_bias |
|:-----------|:-----------|------------:|---------------:|--------------:|----------------:|
| 2regime_96 | <200k | 44 | 0.0533 | 0.7375 | 0.0190 |
| 2regime_96 | 1M+ | 2 | 0.0489 | 0.7565 | 0.0202 |
| 2regime_54 | 200-500k | 21 | 0.0613 | 0.7859 | 0.0033 |
| 2regime_54 | 500k-1M | 8 | 0.0643 | 0.7664 | -0.0018 |
| 2regime_54 | 1M+ | 3 | 0.0566 | 0.7713 | 0.0070 |
| 2regime_mixed | <200k | 8 | 0.0548 | 0.7654 | 0.0122 |
| 2regime_mixed | 2

The sweep is sized to spend ~1 h of the 2 h H100 wall allocation: 178 phase-1 + 110 phase-2 + 44 phase-3 + 6 champion job-seeds ≈ 338 jobs at 8 parallel workers (2.0's MLP-only per-seed mean was 63 s at 76 % utilization).

In [13]:
print("### Timing (H100 PCIe 80 GB, 8 parallel workers)")
print(f"Total sweep wall time: {timing_log.get('sweep_wall_s', float('nan')):.1f} s")
print(f"Total training time (all jobs, GPU-seconds): {sum(j.get('train_time_s', 0.0) for j in timing_log.get('jobs', {}).values()):.0f} s")
print(f"Eval wall time: {timing_log.get('eval_wall_s', float('nan')):.1f} s")
print()
slow = sorted(timing_log.get("jobs", {}).items(), key=lambda kv: -(kv[1].get("train_time_s") or 0))[:5]
print("Slowest jobs (3-seed config train_time_s):")
for k, v in slow:
    print(f"  {k:55s} {v.get('train_time_s', float('nan')):8.1f}s  n_seeds={v.get('n_seeds')}")

### Timing (H100 PCIe 80 GB, 8 parallel workers)
Total sweep wall time: 2595.9 s
Total training time (all jobs, GPU-seconds): 15275 s
Eval wall time: 11.3 s

Slowest jobs (3-seed config train_time_s):
  2regime_mixed/w512x512x512_d0.3_huber0.1_swa070            156.7s  n_seeds=3
  2regime_mixed/w512x512x512_d0.3_huber0.1_swa075            156.4s  n_seeds=3
  2regime_mixed/w512x512x512_d0.3_huber0.1                   156.3s  n_seeds=3
  2regime_mixed/w512x512x512_d0.3_huber0.1_swa080            153.8s  n_seeds=3
  2regime_96/w512x512x512_d0.3_huber0.1_swa                  153.7s  n_seeds=3


## Key Takeaways

1. **2.1's optimization round is an honest negative against 2.0's ceiling.**
   The mixed-family huber-delta/lr grid's 3-seed val winner (0.7844) does not
   beat 2.0's 2-seed honest single (0.7903) or its val top-5 ensemble (0.8003);
   the 54/96 winners match 1.3/2.0's val-selected configs. The remaining MLP
   gap to the (test-selected) XGBoost 2-regime 0.815 is not closed by
   architecture/LR/loss tuning of the plain MLP.
2. **SWA is closed out, with proof.** The RNG guard makes the comparison
   exact (136/136 bit-identical), and 0/152 deployments across starts
   0.6–0.85 means SWA adds nothing at this scale. Do not re-spend on SWA.
3. **Val selection is the bottleneck for the 54 and mixed families.**
   Spearman(val, test) is −0.555 (54) / −0.309 (mixed) even at 3-seed
   aggregation, and the winner flips with seed depth. 3-seed aggregation did
   not fix the noise; the honest next step is a held-out val *season* or
   cross-year val selection, not more seeds on the same val split.
4. **Bias control works exactly where it is targeted** (54-family 0.8 %, and
   near-unbiased small nets exist for 96/mixed) but the val-selected winners
   are biased — the <5 % criterion is a per-family *median* over the honest
   pool, which the 96/mixed families still miss (13.9 % / 12.7 %).
5. **The 2.1 deliverables that hold up:** the RNG-guard proof (bit-identity
   stack), the 3-seed selection-reliability data (winner flips documented),
   the 54-family `w320x320_d0.3_gelu_lr6e-4` test-best reference (0.7935),
   and a fully reproducible v8 sweep (178 configs, 338 job-seeds, 43 min).

All numbers above are the stdout of this notebook; weights/checkpoints/test
predictions under `models/`; preprocessed tensors and per-job logs under
`artifacts/`; figures at the experiment root. See README.md for the full
reproducibility checklist and caveats.